# BoC — Rationale-alignment evaluation (LLM-as-a-judge, on the side)

This notebook is a **side-channel** evaluation: it does not touch the resolution
loop. It is **trace-driven** — the Langfuse **trace** is the canonical record of
what each forecaster said. For every trace it reads the structured forecast the
predictor stamped on at run time (its `rationale`, cited signals, and predicted
distribution), compares that rationale to the Bank of Canada's **own** published
press release for that meeting, and **pushes** a structured *alignment* verdict
back to the trace as Langfuse scores — complementing the accuracy score (RPS)
with a *process* metric: was the forecaster right **for the right reasons**?

So evaluation **reads from and writes to Langfuse**, not a local prediction cache.

**Prerequisites**
1. Langfuse configured (`LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` in `.env`).
2. Press releases cached: `uv run python scripts/fetch_boc_press_releases.py`
   (covers every scheduled date back to 2009).
3. A **traced** run of the reasoning predictors so traces exist in Langfuse. The
   generation cell below recomputes when `FORCE_REFRESH = True`; a cached-only run
   has no fresh trace to evaluate (see the note in section 2).

**Leakage caveat.** For 2010–2024 origins the model may have memorised the
decision; alignment there carries the same caveat as the accuracy score. The
protected 2025–26 window is the cleaner read.


---
## 1. Setup

In [1]:
from __future__ import annotations

import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import Markdown, display  # noqa: A004


warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parents[1]
load_dotenv(ROOT / ".env")

from aieng.forecasting.evaluation import BacktestSpec, cached_backtest
from boc_rate_decisions.data import DIRECTION_SERIES_ID, build_boc_service
from boc_rate_decisions.press_releases import PressReleaseStore
from boc_rate_decisions.rationale_eval import evaluate_result_alignment


STATCAN_CACHE = ROOT / "data" / "statcan"
FRED_CACHE = ROOT / "data" / "fred"
PREDICTIONS_DIR = ROOT / "data" / "predictions"
SPECS_DIR = ROOT / "implementations" / "boc_rate_decisions" / "specs"
# Anchor the press-release cache to the repo root (notebook cwd is the use-case dir).
PRESS_RELEASE_CACHE = ROOT / "data" / "reports" / "boc_press_releases"

svc = build_boc_service(statcan_cache_dir=STATCAN_CACHE, fred_cache_dir=FRED_CACHE)
_as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)
direction_df = svc.get_series(DIRECTION_SERIES_ID, as_of=_as_of)

store = PressReleaseStore.from_cache(PRESS_RELEASE_CACHE)
print(f"Cached press releases: {len(store)}")
if len(store) == 0:
    print("No releases cached — run:  uv run python scripts/fetch_boc_press_releases.py")

Cached press releases: 139


---
## 2. Generate the traced runs to evaluate

Only methods that produce a `rationale` (the agent and the reasoning-enabled
LLMP) can be alignment-scored; the baselines are skipped automatically.

The judge reads each forecast **from its Langfuse trace**, so a traced run must
exist. Set `FORCE_REFRESH = True` to (re)run the predictors with tracing on — this
emits fresh traces, each stamped with the structured forecast. A cached-only run
(`FORCE_REFRESH = False`) reuses predictions whose `langfuse_trace_id` points at
traces from an **earlier** traced run; if no such traces exist yet (or they predate
forecast-stamping), section 3 will score 0 and you should refresh.

> The local prediction cache still backs the **accuracy** backtest in
> `02_boc_rate_direction_experiment.ipynb`; here it only supplies the trace ids
> (pointers) — the rationale and distribution come from Langfuse.

In [2]:
from boc_rate_decisions.analyst_agent import build_boc_agent_predictor, build_boc_basic_config
from boc_rate_decisions.predictors import build_llmp_direction


FORCE_REFRESH = True  # True (re)runs the predictors with tracing on, emitting fresh Langfuse traces

with (SPECS_DIR / "boc_rate_direction_smoke.yaml").open() as f:
    spec = BacktestSpec.model_validate(yaml.safe_load(f))

llmp = build_llmp_direction(model="gemini-3.1-flash-lite-preview", reasoning_effort="low")
agent = build_boc_agent_predictor(build_boc_basic_config(model="gemini-3.1-flash-lite-preview"))
PREDICTOR_LABELS = {llmp.predictor_id: "LLMP direction", agent.predictor_id: "Agent (basic)"}

results = {}
for predictor in [llmp, agent]:
    results[predictor.predictor_id] = cached_backtest(
        predictor=predictor,
        spec=spec,
        spec_id="boc_rate_direction_smoke",
        data_service=svc,
        store_dir=PREDICTIONS_DIR,
        force_refresh=FORCE_REFRESH,
    )
print("Loaded results for:", ", ".join(PREDICTOR_LABELS[p] for p in results))

Loaded results for: LLMP direction, Agent (basic)


---
## 3. Judge each trace and push scores

For every trace the evaluator fetches it from Langfuse (polling briefly, since
ingestion is async), reads the stamped forecast, and runs one LLM-as-judge call
(advanced model). The judge scores *alignment only*; correctness comes from the
realised decision, and the two combine into `right_for_right_reasons`. With
`PUSH_TO_LANGFUSE = True` the verdict is written straight back to the trace as a
numeric `rationale_alignment` score and a categorical `right_for_right_reasons`
score, so it shows up alongside the trace in the Langfuse UI.

In [3]:
PUSH_TO_LANGFUSE = True  # write rationale_alignment + right_for_right_reasons scores back to each trace

frames = [
    evaluate_result_alignment(result, store, direction_df, push_to_langfuse=PUSH_TO_LANGFUSE)
    for result in results.values()
]
nonempty = [f for f in frames if not f.empty]
alignment = pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame()

if alignment.empty:
    print(
        "Scored 0 forecasts. Check that (1) the predictors ran with Langfuse tracing on so their traces exist "
        "(set FORCE_REFRESH = True in section 2), and (2) press releases are cached for these meetings "
        "(run scripts/fetch_boc_press_releases.py)."
    )
else:
    alignment["label"] = alignment["predictor_id"].map(PREDICTOR_LABELS)
    print(f"Scored {len(alignment)} rationale-bearing forecast(s).\n")
    summary = alignment.groupby("label").agg(
        n=("alignment_score", "size"),
        mean_alignment=("alignment_score", "mean"),
        correct_aligned=("right_for_right_reasons", lambda s: int((s == "correct_aligned").sum())),
    )
    print(summary.to_string())

Scored 6 rationale-bearing forecast(s).

                n  mean_alignment  correct_aligned
label                                             
Agent (basic)   3        0.916667                3
LLMP direction  3        0.883333                2


---
## 4. Per-meeting verdicts

Rendered as markdown (not crammed into a figure). Each verdict links to its
Langfuse trace when one is available.

In [4]:
if alignment.empty:
    print("Nothing to show — see the message above.")
else:
    for _, row in alignment.sort_values(["meeting_date", "label"]).iterrows():
        signals = ", ".join(row["key_signal_overlap"]) if row["key_signal_overlap"] else "—"
        trace = f"  ·  [trace]({row['langfuse_trace_url']})" if row.get("langfuse_trace_url") else ""
        display(
            Markdown(
                f"**{row['label']} — {row['meeting_date'].date()}**{trace}  \n"
                f"predicted **{row['predicted_label']}** · realised **{row['realized_label']}** · "
                f"alignment **{row['alignment_score']:.2f}** · _{row['right_for_right_reasons']}_\n\n"
                f"Signal overlap: {signals}\n\n"
                f"{row['justification']}\n\n---"
            )
        )

**Agent (basic) — 2024-04-10**  ·  [trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/2e6a48178780be1886db1c12d70a0da9)  
predicted **hold** · realised **hold** · alignment **0.95** · _correct_aligned_

Signal overlap: Policy rate at 5.0%, Inflation gap narrowing toward 2% target, Modest upward trend in unemployment rate

The forecaster's rationale aligns almost perfectly with the Bank of Canada's official narrative, specifically highlighting that the Bank requires sustained downward momentum in core inflation before cutting. The forecaster also accurately identified the gradual rise in the unemployment rate (which the Bank noted reached 6.1% in March) and the narrowing inflation gap as key underlying drivers. The only minor point of divergence is the forecaster's focus on the 2-year yield inversion, a market-pricing signal that the Bank's press release does not explicitly reference as a driver for its decision.

---

**LLMP direction — 2024-04-10**  ·  [trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/c93ffbaa385da8bb1c4c87840998445e)  
predicted **hold** · realised **hold** · alignment **0.85** · _correct_aligned_

Signal overlap: decelerating inflation, CPI data, data-dependent assessment

The forecaster's rationale closely aligns with the Bank of Canada's stance, correctly identifying that the Bank is in a cautious, data-dependent transitionary phase. The Bank's release confirms this by stating that while CPI and core inflation have eased, the Governing Council needs to see evidence that this downward momentum is sustained before adjusting rates. Both the forecaster and the Bank emphasize the critical role of upcoming inflation data in determining the timing of future rate cuts.

---

**Agent (basic) — 2024-06-05**  ·  [trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/247e627e3d099a331b0b50c4646b8389)  
predicted **cut** · realised **cut** · alignment **0.85** · _correct_aligned_

Signal overlap: Rising unemployment momentum indicating slack in the economy, Closing of the inflation gap toward the 2% target

The forecaster's core economic arguments closely align with the Bank of Canada's rationale, specifically pointing to a cooling labour market (which the Bank described as employment growing slower than the working-age population) and easing inflation moving toward the 2% target. However, the Bank's release does not mention market-based pricing or the yield spread, which the forecaster cited as a key signal. Overall, the fundamental drivers of the decision to cut are highly consistent between both parties.

---

**LLMP direction — 2024-06-05**  ·  [trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/9061b1d399ced994f58044b1b8867fb2)  
predicted **hold** · realised **cut** · alignment **0.95** · _incorrect_aligned_

Signal overlap: cooling inflation, sluggish economic growth, restrictive interest rate environment

The forecaster's rationale strongly aligns with the Bank of Canada's decision to cut rates, pointing to cooling inflation and sluggish economic growth as key drivers. The Bank's release confirms these exact points, noting that CPI inflation eased to 2.7% with core measures slowing, and that Q1 GDP growth at 1.7% was slower than previously forecast. Additionally, both the forecaster and the Bank highlight that the highly restrictive interest rate environment is no longer as necessary given these cooling indicators.

---

**Agent (basic) — 2024-09-04**  ·  [trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/96bae024e83edc842c28acc7bafea169)  
predicted **cut** · realised **cut** · alignment **0.95** · _correct_aligned_

Signal overlap: Rising unemployment momentum, Inflation cooling, Consecutive rate cuts

The forecaster's rationale closely aligns with the Bank of Canada's stated reasons for the rate cut, specifically highlighting cooling inflation and a slowing labour market (which the Bank notes as 'the labour market continues to slow' and inflation slowing to 2.5%). Additionally, the forecaster's focus on preventing 'excessive economic slack' matches the Bank's observation that 'excess supply in the economy continues to put downward pressure on inflation'. The forecaster also correctly anticipated the continuation of the easing cycle ('a further 25 basis points' cut).

---

**LLMP direction — 2024-09-04**  ·  [trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/fc9c7c457a43fa76db474bd0fcf1fb84)  
predicted **cut** · realised **cut** · alignment **0.85** · _correct_aligned_

Signal overlap: cooling inflation, economic slack

The forecaster's core economic arguments of 'cooling inflation' and 'economic slack' align closely with the Bank's explanation that 'continued easing in broad inflationary pressures' and 'excess supply in the economy' drove the decision to cut. Although the forecaster also relies on a historical momentum argument regarding consecutive cuts, the fundamental economic drivers cited match the Bank's stated rationale. The Bank's release confirms these exact dynamics, noting that inflation slowed to 2.5% and excess supply is putting downward pressure on prices.

---

---
## 5. Langfuse scores — review

The `rationale_alignment` and `right_for_right_reasons` scores were pushed to each
trace in section 3 (when `PUSH_TO_LANGFUSE = True`). This table summarises what
landed and links to each trace, so the verdicts are one click from the traces and
dashboards — a step toward closing the agent feedback loop.

In [5]:
if alignment.empty:
    print("Nothing scored — see section 3.")
else:
    n_total = len(alignment)
    n_pushed = int(alignment["langfuse_scored"].sum())

    table = ["| Method | Meeting | Pred → Real | Alignment | Pushed | Trace |", "|---|---|---|---:|:--:|---|"]
    for _, row in alignment.sort_values(["meeting_date", "label"]).iterrows():
        link = f"[open trace]({row['langfuse_trace_url']})" if row.get("langfuse_trace_url") else "—"
        pushed = "✅" if row.get("langfuse_scored") else "—"
        table.append(
            f"| {row['label']} | {row['meeting_date'].date()} | "
            f"{row['predicted_label']} → {row['realized_label']} | "
            f"{row['alignment_score']:.2f} | {pushed} | {link} |"
        )

    header = f"**Langfuse — `rationale_alignment`**  \nscored **{n_total}** · pushed **{n_pushed}**"
    display(Markdown(header + "\n\n" + "\n".join(table)))

    if not PUSH_TO_LANGFUSE:
        display(
            Markdown(
                "_`PUSH_TO_LANGFUSE = False` in section 3 — set it `True` to write the scores. "
                "Trace links are clickable either way._"
            )
        )

**Langfuse — `rationale_alignment`**  
scored **6** · pushed **6**

| Method | Meeting | Pred → Real | Alignment | Pushed | Trace |
|---|---|---|---:|:--:|---|
| Agent (basic) | 2024-04-10 | hold → hold | 0.95 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/2e6a48178780be1886db1c12d70a0da9) |
| LLMP direction | 2024-04-10 | hold → hold | 0.85 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/c93ffbaa385da8bb1c4c87840998445e) |
| Agent (basic) | 2024-06-05 | cut → cut | 0.85 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/247e627e3d099a331b0b50c4646b8389) |
| LLMP direction | 2024-06-05 | hold → cut | 0.95 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/9061b1d399ced994f58044b1b8867fb2) |
| Agent (basic) | 2024-09-04 | cut → cut | 0.95 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/96bae024e83edc842c28acc7bafea169) |
| LLMP direction | 2024-09-04 | cut → cut | 0.85 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmobssx3g00buad07gb4o1gh6/traces/fc9c7c457a43fa76db474bd0fcf1fb84) |